# Preparación de datos para el NPP-LSTM

## Objetivo

Reproducir el preprocesamiento utilizado en el
proyecto original de Neural Point Processes.

Etapas:
1. Convertir fechas a horas.
2. Filtrar magnitudes.
3. Dividir entrenamiento y prueba.
4. Construir secuencias de 20 eventos.

Fuente: catálogo sísmico IGP.
Referencia: utils.py, NeuralPP.py y AVN_experiment.py.

In [2]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Localizar la raíz del proyecto
raiz = Path.cwd()

if not (raiz / "data").exists():
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

# Funciones originales de nuestro compañero
from utils import (
    datetime_to_hours,
    truncate_catalog_by_threshold,
    append_burn_in_to_test_set,
    collate_times
)

ruta = raiz / "data/raw/SISMOS_LIMA_FORMATO_MODELO_cat.csv"

catalogo = pd.read_csv(ruta)

print("Registros originales:", len(catalogo))

Registros originales: 2578


In [3]:
# Parámetros del proyecto original
Mcut = 3.5
M0pred = 4.5
time_step = 20

# Convertir las fechas en horas
catalogo = datetime_to_hours(catalogo)

# Eliminar registros incompletos, si existen
catalogo = catalogo.dropna()

# Filtrar los sismos según Mcut
catalogo_filtrado = truncate_catalog_by_threshold(
    catalogo, Mcut
)

# Obtener tiempos y magnitudes
times = np.array(catalogo_filtrado["time"])
mags = np.array(catalogo_filtrado["mw"])

print("Registros después del filtro:", len(times))
print("Primeros tiempos:", times[:5])
print("Primeras magnitudes:", mags[:5])

Registros después del filtro: 2441
Primeros tiempos: [    0.         14610.835      19681.95527778 21129.84138889
 21314.39083333]
Primeras magnitudes: [6.  4.8 5.3 4.7 6. ]


In [4]:
# Corte temporal original para IGP
timeupto = 450000

# Entrenamiento
T_train = times[times < timeupto]
M_train = mags[times < timeupto]

# Prueba
T_test = times[times >= timeupto]
M_test = mags[times >= timeupto]

print("Eventos entrenamiento:", len(T_train))
print("Eventos prueba:", len(T_test))

# Agregar historial previo al conjunto de prueba
T_test, M_test = append_burn_in_to_test_set(
    T_train,
    T_test,
    M_train,
    M_test,
    time_step
)

print("Prueba con historial previo:", len(T_test))

Eventos entrenamiento: 1621
Eventos prueba: 820
Prueba con historial previo: 839


In [5]:
# Intervalos de tiempo entre sismos consecutivos
dT_train = np.ediff1d(T_train)

# Magnitudes alineadas con esos intervalos
dM_train = np.delete(M_train, 0)

n = len(dT_train)

# Construir historiales de 20 eventos
input_RNN_times = np.array([
    dT_train[i:i + time_step]
    for i in range(n - time_step)
]).reshape(n - time_step, time_step, 1)

input_RNN_mags = np.array([
    dM_train[i:i + time_step]
    for i in range(n - time_step)
]).reshape(n - time_step, time_step, 1)

# Variables del evento objetivo
input_CHFN = dT_train[time_step:].reshape(-1, 1)
input_CMFN = dM_train[time_step:].reshape(-1, 1)

# Calcular tiempos hasta eventos objetivo y máscara
dT_targets, train_mask = collate_times(
    dT_train, dM_train, M0pred
)

train_mask = train_mask[time_step:].reshape(-1, 1)

print("Historial de tiempos:", input_RNN_times.shape)
print("Historial de magnitudes:", input_RNN_mags.shape)
print("Tiempo objetivo:", input_CHFN.shape)
print("Magnitud objetivo:", input_CMFN.shape)
print("Máscara:", train_mask.shape)

Historial de tiempos: (1600, 20, 1)
Historial de magnitudes: (1600, 20, 1)
Tiempo objetivo: (1600, 1)
Magnitud objetivo: (1600, 1)
Máscara: (1600, 1)


In [6]:
# Inspeccionar la primera muestra de entrenamiento
i = 0

print("Historial de 20 intervalos en horas:")
print(input_RNN_times[i, :, 0])

print("\nMagnitudes de los 20 eventos del historial:")
print(input_RNN_mags[i, :, 0])

print("\nDatos del siguiente evento:")
print("Intervalo temporal:", input_CHFN[i, 0])
print("Magnitud:", input_CMFN[i, 0])

print("\n¿Alcanza el umbral Mw >= 4.5?")
print(bool(train_mask[i, 0]))

Historial de 20 intervalos en horas:
[1.46108350e+04 5.07112028e+03 1.44788611e+03 1.84549444e+02
 1.77610639e+03 7.01000000e+01 1.08494444e+02 3.08226944e+02
 4.94110278e+02 8.44506194e+03 2.54967972e+03 8.49736806e+03
 5.22667000e+03 8.50027778e+01 1.03691667e+01 1.24545139e+03
 6.27055556e+00 2.53838889e+01 1.32972222e+00 2.37331667e+02]

Magnitudes de los 20 eventos del historial:
[4.8 5.3 4.7 6.  6.7 4.9 6.2 5.7 4.6 5.  5.2 4.8 4.9 4.9 4.8 5.4 4.8 4.9
 4.9 4.8]

Datos del siguiente evento:
Intervalo temporal: 525.1194444444409
Magnitud: 4.7

¿Alcanza el umbral Mw >= 4.5?
True
